In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession \
    .builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()

In [ ]:
# Jointure orders + customers ssur customer_id

PATH = "/home/jovyan/data/tmp"

df_orders = spark.read.parquet(f"{PATH}/orders")
df_customers = spark.read.parquet(f"{PATH}/customers")


df_ord_cust = df_orders.join(df_customers, 
                             on= "customer_id", 
                             how= "left")\
                             .select("order_id", "company_name", "country", "order_date", "freight")

In [ ]:
# Jointure order_details + products , sur product_id

df_order_details = spark.read.parquet(f"{PATH}/order_details")
df_products = spark.read.parquet(f"{PATH}/products")

df_ord_detail_prdct = df_order_details.join(df_products,
                      on = "product_id",
                      how= "left")\
                      .select("product_name", "category_id","unit_price")




In [ ]:
# Jointure products + categories sur category_id

df_products = spark.read.parquet(f"{PATH}/products")
df_categories = spark.read.parquet(f"{PATH}/categories")

df_prdct_catg = df_products.join(df_categories,
                 on="category_id",
                 how="left")\
                 .select("category_name","description")

In [ ]:
# DataFrame enrichi complet
# jointure complète (order_details, orders, customers, products enrichi avec categories, employees, shippers)


df_employees = spark.read.parquet(f"{PATH}/employees")
df_shippers = spark.read.parquet(f"{PATH}/shippers")
df_suppliers = spark.read.parquet(f"{PATH}/suppliers")



df_complet = (
    df_order_details\
    .join(df_orders, on= "order_id")
    .join(df_customers, on= "customer_id")
    .join(df_products, on= "product_id")
    .join(df_categories, on="category_id")
    .join(df_employees, on="employee_id")
    .join(df_shippers, on="shipper_id")
)


In [ ]:
# les colonnes en doublons 
from collections import Counter

{c: n for c, n in Counter(df_complet.columns).items() if n > 1}

{'company_name': 2, 'city': 2, 'country': 2, 'phone': 2}

In [ ]:
# Rename colonne {'company_name': 2, 'city': 2, 'country': 2, 'phone': 2}
df_customers = df_customers.withColumnRenamed("company_name","customer_company_name")

df_shippers = df_shippers.withColumnRenamed("company_name","shipper_company_name")


df_employees = df_employees.withColumnRenamed("city","employee_city")

df_customers = df_customers.withColumnRenamed("city","customer_city")


df_employees = df_employees.withColumnRenamed("country","employee_country")

df_customers = df_customers.withColumnRenamed("country","customer_country")


df_shippers = df_shippers.withColumnRenamed("phone","shipper_phone")

df_customers = df_customers.withColumnRenamed("phone" ,"customer_phone")



In [ ]:
from collections import Counter

df_orders_enriched = (
     df_order_details\
        .join(df_orders, on= "order_id")
        .join(df_customers, on= "customer_id")
        .join(df_products, on= "product_id")
        .join(df_categories, on="category_id")
        .join(df_employees, on="employee_id")
        .join(df_shippers, on="shipper_id")
    )

print("les doublons")
{c: n for c, n in Counter(df_orders_enriched.columns).items() if n > 1}

les doublons


{}

In [ ]:
# CA par client

df_ca_client = (
    df_orders_enriched
    .groupBy("customer_company_name")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca")
    )
    .orderBy(
        F.col("ca").desc()
    )
    .limit(10)
)

df_ca_client.show()

+---------------------+------+
|customer_company_name|    ca|
+---------------------+------+
|           QUICK-Stop|119817|
|   Save-a-lot Markets|119659|
|         Ernst Handel|115238|
|      MÃ¨re Paillarde| 31671|
| Hungry Owl All-Ni...| 27943|
|     HILARION-Abastos| 20546|
| Rattlesnake Canyo...| 20279|
|       Frankenversand| 17864|
|      Folk och fÃ¤ HB| 17559|
|     LINO-Delicateses| 16271|
+---------------------+------+



In [ ]:
# CA par catégorie

df_ca_cat = (
    df_orders_enriched
    .groupBy("category_name")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca"),
        F.countDistinct("product_id").alias("nb_products")
    )
    .orderBy(
        F.col("ca").desc()
    )
)

df_ca_cat.show()

+--------------+------+-----------+
| category_name|    ca|nb_products|
+--------------+------+-----------+
|   Confections|148337|         13|
|       Seafood|145957|         12|
|Dairy Products|136314|          9|
|    Condiments|130148|         11|
|     Beverages|104306|          9|
|Grains/Cereals| 80721|          6|
|       Produce| 54769|          4|
|  Meat/Poultry| 45009|          2|
+--------------+------+-----------+



In [ ]:
# CA par mois

df_ca_month = (
    df_orders_enriched
    .withColumn("mois", F.date_trunc("month", F.col("order_date")))
    .withColumn("year", F.date_trunc("year", F.col("order_date")))
    .groupBy("mois", "year")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca")
    )
    .orderBy("mois", "year")
)

df_ca_month.show()

+-------------------+-------------------+-----+
|               mois|               year|   ca|
+-------------------+-------------------+-----+
|1997-01-01 00:00:00|1997-01-01 00:00:00|83670|
|1997-02-01 00:00:00|1997-01-01 00:00:00|87202|
|1997-03-01 00:00:00|1997-01-01 00:00:00|43075|
|1997-04-01 00:00:00|1997-01-01 00:00:00|73436|
|1997-05-01 00:00:00|1997-01-01 00:00:00|68666|
|1997-06-01 00:00:00|1997-01-01 00:00:00|39652|
|1997-07-01 00:00:00|1997-01-01 00:00:00|78188|
|1997-08-01 00:00:00|1997-01-01 00:00:00|54238|
|1997-09-01 00:00:00|1997-01-01 00:00:00|88329|
|1997-10-01 00:00:00|1997-01-01 00:00:00|84184|
|1997-11-01 00:00:00|1997-01-01 00:00:00|50485|
|1997-12-01 00:00:00|1997-01-01 00:00:00|94436|
+-------------------+-------------------+-----+



In [ ]:

df_comm_employees = (
    df_orders_enriched
    .groupBy("full_name")
    .agg(
        F.countDistinct("order_id").alias("nb_comm_traitees"),
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca"),
        F.round(F.avg(F.datediff(F.col("shipped_date"), F.col("order_date"))), 1).alias("delai_moy_livraison")
    )
    .orderBy(
        F.col("ca").desc()
    )
)

df_comm_employees.show()

+----------------+----------------+------+-------------------+
|       full_name|nb_comm_traitees|    ca|delai_moy_livraison|
+----------------+----------------+------+-------------------+
|Margaret Peacock|              75|175035|                8.3|
| Janet Leverling|              71|149034|                8.9|
|   Nancy Davolio|              54|135507|                7.8|
|   Andrew Fuller|              40| 98639|               10.2|
|     Robert King|              33| 86493|                9.8|
|  Laura Callahan|              53| 84163|                8.0|
|  Michael Suyama|              33| 57898|                7.9|
| Steven Buchanan|              18| 35056|                6.5|
|  Anne Dodsworth|              18| 23736|               10.0|
+----------------+----------------+------+-------------------+



In [ ]:
# Window functions - Rang

from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

window = Window.partitionBy("category_name").orderBy(F.col("ca").desc())

df_ca_rank = (
    df_orders_enriched
    .groupBy("category_name", "product_name")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca")
    )
    .withColumn("dense_rank", F.dense_rank().over(window))
)

df_ca_rank.show()

+-------------+--------------------+-----+----------+
|category_name|        product_name|   ca|dense_rank|
+-------------+--------------------+-----+----------+
|    Beverages|RhÃ¶nbrÃ¤u Kloste...|20744|         1|
|    Beverages|    Chartreuse verte|19471|         2|
|    Beverages|      LakkalikÃ¶Ã¶ri|16671|         3|
|    Beverages|       Outback Lager|14237|         4|
|    Beverages|      Steeleye Stout|10822|         5|
|    Beverages|         Ipoh Coffee| 8596|         6|
|    Beverages|      CÃ´te de Blaye| 7517|         7|
|    Beverages|       Sasquatch Ale| 4665|         8|
|    Beverages|Laughing Lumberja...| 1583|         9|
|   Condiments|     Sirop d'Ã©rable|26056|         1|
|   Condiments|Louisiana Fiery H...|20134|         2|
|   Condiments|        Gula Malacca|18808|         3|
|   Condiments|Original Frankfur...|17652|         4|
|   Condiments|Louisiana Hot Spi...|10392|         5|
|   Condiments|Chef Anton's Caju...| 9686|         6|
|   Condiments|       Anisee

In [ ]:
# Window functions - Cumul

window_2 = Window.orderBy("year", "mois")

df_ca_cumul = (
    df_ca_month
    .withColumn("ca_cumul", F.sum("ca").over(window_2))

)
df_ca_cumul.show()

+-------------------+-------------------+-----+--------+
|               mois|               year|   ca|ca_cumul|
+-------------------+-------------------+-----+--------+
|1997-01-01 00:00:00|1997-01-01 00:00:00|83670|   83670|
|1997-02-01 00:00:00|1997-01-01 00:00:00|87202|  170872|
|1997-03-01 00:00:00|1997-01-01 00:00:00|43075|  213947|
|1997-04-01 00:00:00|1997-01-01 00:00:00|73436|  287383|
|1997-05-01 00:00:00|1997-01-01 00:00:00|68666|  356049|
|1997-06-01 00:00:00|1997-01-01 00:00:00|39652|  395701|
|1997-07-01 00:00:00|1997-01-01 00:00:00|78188|  473889|
|1997-08-01 00:00:00|1997-01-01 00:00:00|54238|  528127|
|1997-09-01 00:00:00|1997-01-01 00:00:00|88329|  616456|
|1997-10-01 00:00:00|1997-01-01 00:00:00|84184|  700640|
|1997-11-01 00:00:00|1997-01-01 00:00:00|50485|  751125|
|1997-12-01 00:00:00|1997-01-01 00:00:00|94436|  845561|
+-------------------+-------------------+-----+--------+



In [ ]:
# Tri et limites

product_vendu = (
    df_orders_enriched
    .groupBy("product_id", "product_name")
    .agg(
        F.sum(F.col("quantite")).alias("nb_prod_vendu")
    )
    .orderBy(F.col("nb_prod_vendu").desc())
    .limit(5)
)

product_vendu.show()

+----------+--------------------+-------------+
|product_id|        product_name|nb_prod_vendu|
+----------+--------------------+-------------+
|        56|Gnocchi di nonna ...|          971|
|        59|Raclette Courdavault|          752|
|        60|   Camembert Pierrot|          665|
|        75|RhÃ¶nbrÃ¤u Kloste...|          630|
|        21| Sir Rodney's Scones|          610|
+----------+--------------------+-------------+



In [ ]:
customer_country = (
    df_orders_enriched
    .groupBy("customer_country")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca")
    )
    .orderBy(F.col("ca").desc())
    .limit(3)
)

customer_country.show()

+----------------+------+
|customer_country|    ca|
+----------------+------+
|         GERMANY|181089|
|             USA|175783|
|         AUSTRIA|123512|
+----------------+------+



In [ ]:
# Ecriture en parquet 

PATH_2 = "/home/jovyan/data/output"

df_orders_enriched.write.mode("overwrite").parquet(f"{PATH_2}/df_orders_enriched")

In [ ]:
# Relecture du fichier Parquet et affichage de nb de lignes

df_orders_enriched_2 = spark.read.parquet(f"{PATH_2}/df_orders_enriched")
df_orders_enriched_2.count()

893

In [ ]:
print(f" nb de lignes apres parquet,  {df_orders_enriched_2.count()}")

 nb de lignes apres parquet,  893


In [ ]:
print(f"nombre de lignes avant le parquet : {df_orders_enriched.count()}")

nombre de lignes avant le parquet : 893


In [ ]:
df_orders_enriched_2.printSchema()

root
 |-- shipper_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- prix_unitaire: integer (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- sous_total: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)
 |-- is_shipped: boolean (nullable = true)
 |-- customer_company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-

In [ ]:
# Comparaison:  CSV vs Parquet

# Le fichier Parquet est plus léger que les CSV

In [ ]:
# 35 : Partitionnement 

df_orders_enriched.write.mode("overwrite").partitionBy("customer_country").parquet(f"{PATH_2}/df_orders_enriched_country")


In [ ]:
# Chargement PostgreSQL via JDBC


jdbc_url = "jdbc:postgresql://postgres:5432/tradecorp_db"

(
    df_orders_enriched
    .write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "orders_enriched")
    .option("user", "tradecorp")
    .option("password", "tradecorp")
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)